In [1]:
import numpy as np
from pathlib import Path
import pandas as pd

print("✓ Libraries imported")

✓ Libraries imported


## Configuration

In [2]:
# Data directory
data_dir = Path('../../data/preprocessed')

# Test files to verify
test_files = [
    'breast_cancer_test.npz',
    'colon_cancer_test.npz',
    'kidney_cancer_test.npz',
    'lung_cancer_test.npz'
]

print(f"Data directory: {data_dir.absolute()}")

Data directory: c:\Users\samue\Capstone_Project\domain-adaptation-multi-cancer\domain-adaptation-multi-cancer-main\notebooks\eda\..\..\data\preprocessed


## Verification Functions

In [3]:
def load_and_check_dataset(file_path):
    """Load a dataset and return its statistics."""
    if not file_path.exists():
        return None
    
    data = np.load(str(file_path))
    
    return {
        'path': file_path.name,
        'total_samples': len(data['images']),
        'image_shape': data['images'].shape,
        'labels1_unique': np.unique(data['labels1']),
        'labels2_unique': np.unique(data['labels2']),
        'labels2_0_count': (data['labels2'] == 0).sum(),
        'labels2_1_count': (data['labels2'] == 1).sum(),
        'labels1_0_count': (data['labels1'] == 0).sum(),
        'labels1_1_count': (data['labels1'] == 1).sum(),
    }

def verify_split(original_stats, target_stats, source_stats):
    """Verify that the split was done correctly."""
    checks = []
    
    # Check 1: Total samples should match
    total_split = target_stats['total_samples'] + source_stats['total_samples']
    original_total = original_stats['total_samples']
    checks.append({
        'check': 'Total samples match',
        'expected': original_total,
        'actual': total_split,
        'pass': total_split == original_total
    })
    
    # Check 2: Target file should only have labels2=1
    checks.append({
        'check': 'Target file only has labels2=1',
        'expected': [1],
        'actual': target_stats['labels2_unique'].tolist(),
        'pass': np.array_equal(target_stats['labels2_unique'], [1])
    })
    
    # Check 3: Source file should only have labels2=0
    checks.append({
        'check': 'Source file only has labels2=0',
        'expected': [0],
        'actual': source_stats['labels2_unique'].tolist(),
        'pass': np.array_equal(source_stats['labels2_unique'], [0])
    })
    
    # Check 4: Target count matches
    checks.append({
        'check': 'Target count matches',
        'expected': original_stats['labels2_1_count'],
        'actual': target_stats['total_samples'],
        'pass': original_stats['labels2_1_count'] == target_stats['total_samples']
    })
    
    # Check 5: Source count matches
    checks.append({
        'check': 'Source count matches',
        'expected': original_stats['labels2_0_count'],
        'actual': source_stats['total_samples'],
        'pass': original_stats['labels2_0_count'] == source_stats['total_samples']
    })
    
    # Check 6: Image shapes match
    checks.append({
        'check': 'Image shapes consistent',
        'expected': original_stats['image_shape'][1:],
        'actual': f"Target: {target_stats['image_shape'][1:]}, Source: {source_stats['image_shape'][1:]}",
        'pass': (target_stats['image_shape'][1:] == original_stats['image_shape'][1:] and 
                 source_stats['image_shape'][1:] == original_stats['image_shape'][1:])
    })
    
    return checks

print("✓ Functions defined")

✓ Functions defined


## Verify All Datasets

In [4]:
all_results = {}

for test_file in test_files:
    print("="*80)
    print(f"Verifying: {test_file}")
    print("="*80)
    
    # Construct file paths
    original_path = data_dir / test_file
    base_name = test_file.replace('.npz', '')
    target_path = data_dir / f"{base_name}_target_only.npz"
    source_path = data_dir / f"{base_name}_source_only.npz"
    
    # Check if files exist
    if not original_path.exists():
        print(f"❌ Original file not found: {original_path}")
        continue
    
    if not target_path.exists():
        print(f"❌ Target file not found: {target_path}")
        continue
    
    if not source_path.exists():
        print(f"❌ Source file not found: {source_path}")
        continue
    
    # Load all three files
    print("\nLoading datasets...")
    original_stats = load_and_check_dataset(original_path)
    target_stats = load_and_check_dataset(target_path)
    source_stats = load_and_check_dataset(source_path)
    
    # Print statistics
    print(f"\nOriginal dataset:")
    print(f"  Total samples: {original_stats['total_samples']}")
    print(f"  Target domain (labels2=1): {original_stats['labels2_1_count']}")
    print(f"  Source domain (labels2=0): {original_stats['labels2_0_count']}")
    print(f"  Benign (labels1=0): {original_stats['labels1_0_count']}")
    print(f"  Malignant (labels1=1): {original_stats['labels1_1_count']}")
    
    print(f"\nTarget-only dataset:")
    print(f"  Total samples: {target_stats['total_samples']}")
    print(f"  Labels2 values: {target_stats['labels2_unique']}")
    print(f"  Benign (labels1=0): {target_stats['labels1_0_count']}")
    print(f"  Malignant (labels1=1): {target_stats['labels1_1_count']}")
    
    print(f"\nSource-only dataset:")
    print(f"  Total samples: {source_stats['total_samples']}")
    print(f"  Labels2 values: {source_stats['labels2_unique']}")
    print(f"  Benign (labels1=0): {source_stats['labels1_0_count']}")
    print(f"  Malignant (labels1=1): {source_stats['labels1_1_count']}")
    
    # Run verification checks
    checks = verify_split(original_stats, target_stats, source_stats)
    
    print("\nVerification Checks:")
    all_passed = True
    for check in checks:
        status = "✓" if check['pass'] else "❌"
        print(f"  {status} {check['check']}")
        if not check['pass']:
            print(f"      Expected: {check['expected']}")
            print(f"      Actual: {check['actual']}")
            all_passed = False
    
    if all_passed:
        print("\n✅ All checks passed!")
    else:
        print("\n❌ Some checks failed!")
    
    # Store results
    all_results[test_file] = {
        'original': original_stats,
        'target': target_stats,
        'source': source_stats,
        'checks': checks,
        'all_passed': all_passed
    }
    
    print()

print("="*80)
print("Verification complete!")
print("="*80)

Verifying: breast_cancer_test.npz

Loading datasets...

Original dataset:
  Total samples: 6000
  Target domain (labels2=1): 1471
  Source domain (labels2=0): 4529
  Benign (labels1=0): 3000
  Malignant (labels1=1): 3000

Target-only dataset:
  Total samples: 1471
  Labels2 values: [1]
  Benign (labels1=0): 737
  Malignant (labels1=1): 734

Source-only dataset:
  Total samples: 4529
  Labels2 values: [0]
  Benign (labels1=0): 2263
  Malignant (labels1=1): 2266

Verification Checks:
  ✓ Total samples match
  ✓ Target file only has labels2=1
  ✓ Source file only has labels2=0
  ✓ Target count matches
  ✓ Source count matches
  ✓ Image shapes consistent

✅ All checks passed!

Verifying: colon_cancer_test.npz

Loading datasets...

Original dataset:
  Total samples: 6000
  Target domain (labels2=1): 1457
  Source domain (labels2=0): 4543
  Benign (labels1=0): 3000
  Malignant (labels1=1): 3000

Target-only dataset:
  Total samples: 1457
  Labels2 values: [1]
  Benign (labels1=0): 730
  Mali

## Summary Table

In [5]:
# Create summary table
summary_data = []

for test_file, results in all_results.items():
    dataset_name = test_file.replace('_test.npz', '').replace('_', ' ').title()
    
    summary_data.append({
        'Dataset': dataset_name,
        'Original Total': results['original']['total_samples'],
        'Target Samples': results['target']['total_samples'],
        'Source Samples': results['source']['total_samples'],
        'Target %': f"{100 * results['target']['total_samples'] / results['original']['total_samples']:.1f}%",
        'Source %': f"{100 * results['source']['total_samples'] / results['original']['total_samples']:.1f}%",
        'All Checks Passed': '✅' if results['all_passed'] else '❌'
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(summary_df.to_string(index=False))
print()


SUMMARY TABLE
      Dataset  Original Total  Target Samples  Source Samples Target % Source % All Checks Passed
Breast Cancer            6000            1471            4529    24.5%    75.5%                 ✅
 Colon Cancer            6000            1457            4543    24.3%    75.7%                 ✅
Kidney Cancer            6000            1494            4506    24.9%    75.1%                 ✅
  Lung Cancer            6000            1578            4422    26.3%    73.7%                 ✅



## Detailed Breakdown by Domain and Class

In [6]:
# Create detailed breakdown
detailed_data = []

for test_file, results in all_results.items():
    dataset_name = test_file.replace('_test.npz', '').replace('_', ' ').title()
    
    # Original dataset
    detailed_data.append({
        'Dataset': dataset_name,
        'Split': 'Original',
        'Total': results['original']['total_samples'],
        'Benign': results['original']['labels1_0_count'],
        'Malignant': results['original']['labels1_1_count'],
        'Benign %': f"{100 * results['original']['labels1_0_count'] / results['original']['total_samples']:.1f}%",
        'Malignant %': f"{100 * results['original']['labels1_1_count'] / results['original']['total_samples']:.1f}%"
    })
    
    # Target dataset
    detailed_data.append({
        'Dataset': dataset_name,
        'Split': 'Target',
        'Total': results['target']['total_samples'],
        'Benign': results['target']['labels1_0_count'],
        'Malignant': results['target']['labels1_1_count'],
        'Benign %': f"{100 * results['target']['labels1_0_count'] / results['target']['total_samples']:.1f}%",
        'Malignant %': f"{100 * results['target']['labels1_1_count'] / results['target']['total_samples']:.1f}%"
    })
    
    # Source dataset
    detailed_data.append({
        'Dataset': dataset_name,
        'Split': 'Source',
        'Total': results['source']['total_samples'],
        'Benign': results['source']['labels1_0_count'],
        'Malignant': results['source']['labels1_1_count'],
        'Benign %': f"{100 * results['source']['labels1_0_count'] / results['source']['total_samples']:.1f}%",
        'Malignant %': f"{100 * results['source']['labels1_1_count'] / results['source']['total_samples']:.1f}%"
    })

detailed_df = pd.DataFrame(detailed_data)
print("\n" + "="*80)
print("DETAILED CLASS DISTRIBUTION")
print("="*80)
print(detailed_df.to_string(index=False))
print()


DETAILED CLASS DISTRIBUTION
      Dataset    Split  Total  Benign  Malignant Benign % Malignant %
Breast Cancer Original   6000    3000       3000    50.0%       50.0%
Breast Cancer   Target   1471     737        734    50.1%       49.9%
Breast Cancer   Source   4529    2263       2266    50.0%       50.0%
 Colon Cancer Original   6000    3000       3000    50.0%       50.0%
 Colon Cancer   Target   1457     730        727    50.1%       49.9%
 Colon Cancer   Source   4543    2270       2273    50.0%       50.0%
Kidney Cancer Original   6000    3000       3000    50.0%       50.0%
Kidney Cancer   Target   1494     742        752    49.7%       50.3%
Kidney Cancer   Source   4506    2258       2248    50.1%       49.9%
  Lung Cancer Original   6000    3000       3000    50.0%       50.0%
  Lung Cancer   Target   1578     791        787    50.1%       49.9%
  Lung Cancer   Source   4422    2209       2213    50.0%       50.0%



## Final Verdict

In [7]:
all_datasets_passed = all(result['all_passed'] for result in all_results.values())

print("\n" + "="*80)
if all_datasets_passed:
    print("✅ SUCCESS: All datasets split correctly!")
    print("\nThe target-only and source-only files are ready to use.")
else:
    print("❌ FAILURE: Some datasets have issues!")
    print("\nPlease review the verification checks above.")
print("="*80)


✅ SUCCESS: All datasets split correctly!

The target-only and source-only files are ready to use.
